# Task 3 — Distributions and Relationships

**Dataset:** `penguins.csv` — 344 penguins from 3 species on 3 islands  
Columns: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`

**Known missing values:** 2 penguins missing all four measurements; 11 missing `sex`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('data/penguins.csv')
print(f'Shape: {df.shape}')
print(f'\nMissing values per column:')
print(df.isnull().sum())
print(f'\nSpecies: {df["species"].unique()}')

## Handling Missing Values

**Decision:** Drop the 2 penguins missing all four body measurements — they cannot contribute to any chart. Retain the 11 penguins missing `sex` for charts 1–3 (body mass distribution, bill scatter) but exclude them from chart 4 (species × sex comparison), which explicitly requires the `sex` column. In both cases, state the exclusion on the chart.

In [ ]:
# Drop rows missing all measurements
df_clean = df.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'])
print(f'After dropping rows with missing measurements: {len(df_clean)} penguins (dropped {len(df) - len(df_clean)})')

# Separate dataset for sex-dependent charts
df_sex = df_clean.dropna(subset=['sex'])
print(f'After additionally dropping missing sex: {len(df_sex)} penguins (dropped {len(df_clean) - len(df_sex)} more)')

## Step 1 — Distribution of Body Mass (All Penguins)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df_clean['body_mass_g'], bins=25, color='steelblue', edgecolor='white')
ax.set_title('Penguin Body Mass Has Two Humps — Hinting at Two Groups\n(n=342; 2 rows excluded: all measurements missing)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Body Mass (g)')
ax.set_ylabel('Count')
ax.set_xlim(left=0)
ax.axvline(df_clean['body_mass_g'].mean(), color='#E15759', linestyle='--', linewidth=1.5,
           label=f'Mean = {df_clean["body_mass_g"].mean():.0f} g')
ax.legend()

plt.tight_layout()
plt.savefig('task3_body_mass_overall.png', dpi=150, bbox_inches='tight')
plt.show()

**Shape:** The distribution is **bimodal** — there are two distinct humps, one around 3,500 g and one around 4,700–5,000 g. A unimodal normal distribution would have one peak. The bimodal shape immediately signals that the penguins form at least two distinct groups.

**Why a histogram?**  
A histogram is the standard tool for showing the shape of a single continuous variable. It reveals whether the distribution is symmetric or skewed, unimodal or multimodal, and where the bulk of the data lies. A bar chart would be wrong here (bars require categories, not bins of continuous data). A box plot would hide the bimodal shape entirely.

## Step 2 — Body Mass Distribution Split by Species

In [ ]:
species_colors = {'Adelie': '#4E79A7', 'Chinstrap': '#F28E2B', 'Gentoo': '#59A14F'}
species_list = ['Adelie', 'Chinstrap', 'Gentoo']

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True, sharex=True)
fig.suptitle('Gentoo Penguins Are Heavier — They Explain the Second Hump\n(n=342; 2 rows excluded: all measurements missing)',
             fontsize=11, fontweight='bold')

for ax, species in zip(axes, species_list):
    subset = df_clean[df_clean['species'] == species]
    ax.hist(subset['body_mass_g'], bins=20, color=species_colors[species], edgecolor='white', alpha=0.85)
    ax.set_title(f'{species} (n={len(subset)})', fontsize=10)
    ax.set_xlabel('Body Mass (g)')
    ax.set_xlim(2500, 6500)

axes[0].set_ylabel('Count')

plt.tight_layout()
plt.savefig('task3_body_mass_by_species.png', dpi=150, bbox_inches='tight')
plt.show()

**Does splitting by species explain the bimodal shape?**  
Yes. Adelie and Chinstrap penguins cluster around 3,500–3,800 g (the first hump), while Gentoo penguins average around 4,700–5,200 g (the second hump). Each species individually has a roughly unimodal, symmetric distribution. The bimodal shape in the overall histogram was entirely caused by mixing two populations with different means — a classic example of why aggregating across groups can mislead.

## Step 3 — Bill Length vs Bill Depth (Simpson's Paradox)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel A: All penguins, no species colour ---
ax = axes[0]
ax.scatter(df_clean['bill_length_mm'], df_clean['bill_depth_mm'],
           color='steelblue', s=20, alpha=0.5)

# Fit overall regression line
m, b = np.polyfit(df_clean['bill_length_mm'], df_clean['bill_depth_mm'], 1)
x_range = np.linspace(df_clean['bill_length_mm'].min(), df_clean['bill_length_mm'].max(), 100)
ax.plot(x_range, m * x_range + b, color='#E15759', linewidth=2,
        label=f'Overall slope = {m:.2f} (negative)')
ax.set_title('Overall: Bill Depth Decreases as Bill Length Grows\n(All species pooled)', fontsize=10, fontweight='bold')
ax.set_xlabel('Bill Length (mm)')
ax.set_ylabel('Bill Depth (mm)')
ax.legend(fontsize=8)

# --- Panel B: Coloured by species ---
ax = axes[1]
patches = []
for species in species_list:
    subset = df_clean[df_clean['species'] == species]
    ax.scatter(subset['bill_length_mm'], subset['bill_depth_mm'],
               color=species_colors[species], s=20, alpha=0.65, label=species)
    # Per-species regression
    m_s, b_s = np.polyfit(subset['bill_length_mm'], subset['bill_depth_mm'], 1)
    x_s = np.linspace(subset['bill_length_mm'].min(), subset['bill_length_mm'].max(), 50)
    ax.plot(x_s, m_s * x_s + b_s, color=species_colors[species], linewidth=2)

ax.set_title('Within Each Species: Bill Depth Increases with Bill Length\n(Simpson\'s Paradox — colour by species)', fontsize=10, fontweight='bold')
ax.set_xlabel('Bill Length (mm)')
ax.set_ylabel('Bill Depth (mm)')
ax.legend(title='Species', fontsize=8)

fig.suptitle('Simpson\'s Paradox: The Aggregate Relationship is the Opposite of the Within-Group Relationship\n(n=342; 2 rows excluded: all measurements missing)',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('task3_simpsons_paradox.png', dpi=150, bbox_inches='tight')
plt.show()

**What is going on — Simpson's Paradox:**  
When all 342 penguins are plotted together (left panel), the regression line has a negative slope: as bill length increases, bill depth decreases. This appears to mean that longer bills are shallower. But when the same data is split by species (right panel), every species shows a *positive* slope — within Adelie, within Chinstrap, and within Gentoo, longer bills are also deeper.

The paradox arises because the three species occupy different regions of the plot. Adelie penguins have short but deep bills; Gentoo penguins have long but shallow bills; Chinstrap are in between. The negative slope in the pooled data is produced entirely by the *between-species* difference in average bill shape, not by a genuine within-species relationship. When you pool the groups, the species-level confound overwhelms the within-species signal and reverses the apparent direction.

This is the best argument for always plotting groups separately before combining them. An aggregated correlation coefficient would point you in the wrong direction.

**Why scatter plots?**  
A scatter plot is the only chart that simultaneously shows the relationship between two continuous variables and lets colour encode a third categorical variable (species). A grouped bar chart could show means but would hide the continuous relationship and individual variation.

## Step 4 — Average Body Mass by Species and Sex

In [ ]:
avg_mass = df_sex.groupby(['species', 'sex'])['body_mass_g'].mean().reset_index()
print(avg_mass)

# Grouped bar chart
species_order = ['Adelie', 'Chinstrap', 'Gentoo']
sexes = ['MALE', 'FEMALE']
sex_colors = {'MALE': '#4E79A7', 'FEMALE': '#E15759'}
sex_labels = {'MALE': 'Male', 'FEMALE': 'Female'}

x = np.arange(len(species_order))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 6))

for i, sex in enumerate(sexes):
    vals = [avg_mass[(avg_mass['species'] == sp) & (avg_mass['sex'] == sex)]['body_mass_g'].values[0]
            for sp in species_order]
    bars = ax.bar(x + (i - 0.5) * width, vals,
                  width=width, color=sex_colors[sex], label=sex_labels[sex], edgecolor='white')
    # Label bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                f'{val:.0f}g', ha='center', va='bottom', fontsize=8)

ax.set_title('Male Gentoo Penguins Are the Heaviest; All Males Outweigh Females in the Same Species\n(n=333; 2 rows excluded: missing measurements; 11 excluded: missing sex)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Species')
ax.set_ylabel('Average Body Mass (g)')
ax.set_xticks(x)
ax.set_xticklabels(species_order)
ax.set_ylim(0, 6000)
ax.legend(title='Sex')

plt.tight_layout()
plt.savefig('task3_body_mass_species_sex.png', dpi=150, bbox_inches='tight')
plt.show()

**Why a grouped bar chart?**  
This question asks for a *comparison* across two categorical dimensions simultaneously (species and sex). A grouped bar chart is designed for exactly this — it places bars side by side within each species group, making within-species male/female comparison direct (adjacent bars), while the groupings make across-species comparison easy. Bars start at zero because this is a count/magnitude, not a difference. Colour encodes sex (the secondary grouping variable), which is meaningful and purposeful.